## Goal
1. The USAE screwed up Softmax distributions. We will now verify if
    - Single SAE SGen Score Happens
    - Systematically, one by one, see if training a separate checkpoint leads to anything meaningful.

Setup

In [ ]:
import os
import torch
import warnings
import torch.nn as nn
from tqdm import tqdm
from einops import rearrange
import matplotlib.pyplot as plt
import torch.nn.functional as F
from overcomplete.sae import TopKSAE
from domainbed.networks import Identity
from timm.layers import SelectAdaptivePool2d
from domainbed.algorithms import DANN, CORAL, Mixup, MMD, IRM, ERM, SagNet
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR


from lib.loaders import load_backbone
from lib.parser import get_top_k_steps
from lib.utils import extract_features
from lib.data_handlers import  Load_PACS
from lib.gpu_pacs import get_pacs_gpuloader, get_pacs_standard_loader



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
algo_classes = {"DANN": DANN, "CORAL": CORAL, "Mixup": Mixup, "MMD": MMD, "IRM": IRM, "ERM": ERM, "SagNet": SagNet}
warnings.filterwarnings("ignore", category=UserWarning)
torch.cuda.empty_cache()

Load the Models

In [ ]:
model_dir = r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\PACS_ResNet_Sketch_Test_Only\ERM_ResNet_T3"
best_nonoracle_steps = get_top_k_steps(os.path.join(model_dir, "out.txt"), envs=[0, 1, 2], k=1)
best_oracle_steps = get_top_k_steps(os.path.join(model_dir, "out.txt"), envs=[0, 1, 2, 3], k=1)


print(f"Best Non-Oracle: {best_nonoracle_steps}")
print(f"Best Oracle: {best_oracle_steps}")

backbones = {}
for step in set(best_nonoracle_steps).union(set(best_oracle_steps)):
    backbones[step] = load_backbone("ERM_ResNet_T3", os.path.join(model_dir, f"model_step{step}.pkl"))

In [ ]:
class Normalizer(nn.Module): 
    def __init__(self, model, dataset, domains=None):
        super().__init__() 

        self.dataset = dataset
        if self.dataset == 'PACS':
            dl, _ = Load_PACS(domains=domains, batch_size=1024)
            x, _ = next(iter(dl))
            
            model.to(device)
            activations = extract_features(model, x.to(device))

        flat = activations.flatten()
        
        self.register_buffer('mean', flat.mean())
        self.register_buffer('std', flat.std())
                
    def forward(self, activations): 
        activations = (activations - self.mean)
        activations = activations / (self.std + 1e-12)
        return activations

    def denormalize(self, normalized_activations):
        return (normalized_activations * (self.std + 1e-12)) + self.mean

In [ ]:
domains = {
    0: "art_painting", 
    1: "cartoon",
    2: "photo",
    3: "sketch"
}

# TEST TO TRAIN ENVS
envs = {
    "T0": [1, 2, 3],
    "T1": [0, 2, 3],
    "T2": [0, 1, 3],
    "T3": [0, 1, 2],
    "T23": [0, 1], 
    "T13": [0, 2], 
    "T12": [0, 3], 
    "T03": [1, 2], 
    "T02": [1, 3], 
    "T01": [2, 3]
} 


def train_pacs_USAEs(backbones, 
                     save_path, 
                     trainenvs, 
                     rearrange_string='n t d -> (n t) d', 
                     nb_concepts=None, 
                     top_k=None, 
                     learning_rate=None, 
                     epochs=None, 
                     checkpoint_path=None, 
                     batch_size=64, 
                     full_data_gpu=False,
                     enable_logging=False, 
                     logging_dir=None, 
                     name_flag=None,
                     # Added missing arguments used in string formatting
                     name="USAE",
                     algo_name="ERM",
                     backbone_name="Multi",
                     test_envs="test"):
    """
    Train a Sparse Autoencoder on PACS data.
    """
        
    # Choose loader based on full_data_gpu flag
    if full_data_gpu:
        domain_train_loader = get_pacs_gpuloader(domains=[domains[e] for e in trainenvs], batch_size=batch_size, drop_last=True)
    else:
        domain_train_loader = get_pacs_standard_loader(domains=[domains[e] for e in trainenvs], batch_size=batch_size, drop_last=True)

    print(f"train envs: {trainenvs}")
    
    # Validate SAE configuration
    if nb_concepts is None or top_k is None:
        raise ValueError(
            "SAE configuration (nb_concepts and top_k) must be explicitly provided via command-line arguments. "
            "No defaults are assumed. Use --nb-concepts and --top-k to specify values."
        )
    
    # Set defaults for learning rate and epochs if not provided
    if learning_rate is None:
        learning_rate = 3e-4
    if epochs is None:
        epochs = 250
    
    print(f"SAE Configuration: nb_concepts={nb_concepts}, top_k={top_k}")
    print(f"Training Configuration: lr={learning_rate}, epochs={epochs}")
    
    # Initialize or load SAE
    SAEs = {}
    if checkpoint_path is not None:
        for key in backbones.keys():
            checkpoints = torch.load(checkpoint_path, weights_only=False)
            SAEs[key] = checkpoints[key]
            SAEs[key].train()
            if not hasattr(SAEs[key], "normalizer"):
                SAEs[key].normalizer = Normalizer(backbones[key], "PACS")

            # Assuming device is defined globally, otherwise add to args
            SAEs[key].to(device)
            backbones[key].to(device)
    
    else:
        fdim = nb_concepts // 8
        for key in backbones.keys():
            SAEs[key] = TopKSAE(fdim, nb_concepts=nb_concepts, top_k=top_k, device="cuda")
            SAEs[key].train()

            if not hasattr(SAEs[key], "normalizer"):
                SAEs[key].normalizer = Normalizer(backbones[key], "PACS")
            backbones[key].to(device)


    optimizers = {}
    schedulers = {}

    for key in backbones.keys():
        optimizers[key] = torch.optim.Adam(SAEs[key].parameters(), lr=learning_rate)
        warmup_scheduler = LinearLR(optimizers[key], start_factor=1e-6 / learning_rate, end_factor=1.0, total_iters=10)
        cosine_scheduler = CosineAnnealingLR(optimizers[key], T_max=epochs-25, eta_min=1e-6)
        schedulers[key] = SequentialLR(optimizers[key], schedulers=[warmup_scheduler, cosine_scheduler], milestones=[25])
    
    criterion = nn.L1Loss(reduction="mean")  
    
    # Validate logging configuration and load history if resuming
    loss_history = []
    starting_epoch = 0
    if enable_logging:
        if logging_dir is None:
            raise ValueError("logging_dir must be provided when enable_logging=True")
        os.makedirs(logging_dir, exist_ok=True)
        loss_log_file = os.path.join(logging_dir, f"loss_{algo_name}_{backbone_name}_{test_envs}" + (f"_{name_flag}" if name_flag else "") + ".txt")
        loss_curve_file = os.path.join(logging_dir, f"loss_curve_{algo_name}_{backbone_name}_{test_envs}" + (f"_{name_flag}" if name_flag else "") + ".png")
        
        # Load previous loss history if resuming from checkpoint
        if checkpoint_path is not None and os.path.exists(loss_log_file):
            print(f"Loading existing loss history from checkpoint resume...")
            with open(loss_log_file, 'r') as f:
                loss_lines = f.readlines()
            for line in loss_lines:
                try:
                    loss_value = float(line.split(': ')[-1].strip())
                    loss_history.append(loss_value)
                except:
                    pass
            starting_epoch = len(loss_history)
            print(f"Resumed from epoch {starting_epoch}. Will train for {epochs} more epochs.")
    else:
        loss_log_file = None
        loss_curve_file = None

    # FIX: Initialize rotator
    rotator = 0

    pbar = tqdm(range(epochs), desc=f"Training SAE: {name}")
    for epoch in pbar:
        actual_epoch_num = starting_epoch + epoch + 1  # For display (1-indexed)
        total_epochs = starting_epoch + epochs
        epoch_loss = 0.0
        
        for i, (images, _) in enumerate(domain_train_loader):
            total_loss = 0.0
            names = list(optimizers.keys()) # FIX: Need list() to index dict keys
            
            images = images.to(device)

            for k in names:
                optimizers[k].zero_grad()
                
            # Current SAE
            current = names[rotator]

            # Current SAE model
            backbone = backbones[current]
            sae = SAEs[current]
            sae.train()
            
            # Encoder Forward Pass
            x = extract_features(backbone, images) # Forward Pass
            x = sae.normalizer(x) # Normalize
            x = rearrange(x, rearrange_string) # Rearrange
            _, z = sae.encode(x)

            # Decoder across all models & accumulate loss
            for n, m in SAEs.items():
                if n == current:
                    x_hat = m.decode(z)
                else:
                    x_hat = m.decode(z.detach())

                loss = criterion(x_hat, x)
                total_loss += loss

            total_loss.backward()
            
            optimizers[current].step()
            if schedulers:
                schedulers[current].step()

            # Rotator Update
            rotator += 1
            rotator = rotator % len(names)

            # FIX: Accumulate batch loss for logging!
            epoch_loss += total_loss.item()
            
        # Average loss over batches
        avg_epoch_loss = epoch_loss / len(domain_train_loader)
        loss_history.append(avg_epoch_loss)
        
        # Log and update plot if logging is enabled
        if enable_logging:
            # Write to log file (appends to existing)
            with open(loss_log_file, 'a') as f:
                f.write(f"Epoch {actual_epoch_num}/{total_epochs}: {avg_epoch_loss:.6f}\n")
            
            # Update loss curve every 10 epochs or at the end
            if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
                plt.figure(figsize=(10, 6))
                plt.plot(loss_history, linewidth=2)
                plt.xlabel('Epoch')
                plt.ylabel('Loss')
                plt.title(f'Training Loss - {name}')
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(loss_curve_file, dpi=100)
                plt.close() # Important to close so we don't leak memory
        
        # Update tqdm with loss info
        pbar.set_postfix(loss=f"{avg_epoch_loss:.6f}")

    # FIX: Save the entire dictionary, not just the last SAE, to match loading logic
    final_save_path = os.path.join(save_path, f"USAE_{algo_name}_{backbone_name}_{test_envs}" + (f"_{name_flag}" if name_flag else "") + ".pt")
    torch.save(SAEs, final_save_path)
    print(f"SAEs saved successfully to {final_save_path}")

In [ ]:
train_pacs_USAEs(
    backbones,
    r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\SAEs\normalization_testing", 
    trainenvs=[0, 1, 2],
    nb_concepts=2048*8,
    top_k=16,
    learning_rate=3e-4,
    epochs=250,
    checkpoint_path=r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\SAEs\normalization_testing\USAE_ERM_Multi_test_3300_2100.pt",
    rearrange_string='n c w h -> (n w h) c',
    batch_size=64,
    full_data_gpu=True,
    enable_logging=True,
    logging_dir=r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\logs\normal_USAE_loaded",
    name_flag="3300_2100"
)

Train a single SAE on 3300

In [ ]:
SAEs = torch.load(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\SAEs\normalization_testing\USAE_ERM_Multi_test_3300_2100.pt", weights_only=False)
print(SAEs.keys())

Running Analysis

In [ ]:
import math
import torch
from tqdm import tqdm
from einops import rearrange
from lib.data_handlers import Load_PACS
import json
import os
import json
from collections import defaultdict, Counter
from overcomplete.visualization.plot_utils import (interpolate_cv2, get_image_dimensions, show)
from overcomplete.visualization.cmaps import VIRIDIS_ALPHA
import torch.nn.functional as F


#domains = ["photo", "art_painting", "cartoon", "sketch"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def calculate_mean_activations(backbone, sae, rearrange_string, w=14, domains=domains, nb_concepts=7680):
    backbone.eval()
    sae.eval()
    activations = {}
    for cls in range(7):
        
        activations[cls] = {}
        z_d = torch.zeros((len(domains), nb_concepts)).to(device)
        
        ### Added Counts
        counts = torch.zeros((len(domains),)).to(device)  # ADD: track sample counts per domain

        for d, domain in enumerate(domains):
            loader, _ = Load_PACS(domains=[domain])
            for i, batch in enumerate(tqdm(loader)):
                with torch.no_grad():
                    img, y = batch
                    img, y = img.to(device), y.to(device)
                    
                    x = extract_features(backbone, img)
                    x = sae.normalizer(x)
                    x = rearrange(x, rearrange_string)

                    _, heatmaps = sae.encode(x)

                    mask = (y == cls).squeeze().to(device)  # (batch_size,)
                    heatmaps = rearrange(heatmaps, '(n w h) d -> n w h d', w=w, h=w)  # (n, t, d)
                    heatmaps_filtered = heatmaps[mask]  # (n_cls, t, d)
                    
                    z_d[d] += heatmaps_filtered.sum(dim=0).sum(dim=0).sum(dim=0)
                    
                    #### Added counter of number of images added
                    counts[d] += mask.sum()

        ## Divide by Safe Count
        safe_counts = counts.clamp(min=1).unsqueeze(1)
        activations[cls] = z_d / safe_counts
        
    return activations

def save_json(data, filepath):
    try:
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"Successfully saved logs to {filepath}")
    except TypeError as e:
        print(f"Error saving JSON: {e}. Check for non-serializable types (like tensors).")
    except Exception as e:
        print(f"An error occurred: {e}")

def calculate_invariance(activations, ent_thresh=0.7, act_thresh=0.0, domains=domains, nb_concepts=7680):
    clss = 7
    logs = {
        "model_invariance" : 0,
        "final_invariance_per_class": {},
        "thresholded_concept_entropies": {}
    }
    for cls in range(clss):
        # mask = probabilities[cls] != 0.25
        processed = activations[cls] # * mask
        
        sum_entropy = 0.0
        class_concept_logs = []

        for i in range(nb_concepts):
            if processed[:, i].sum() == 0:
                continue

            #score = processed[:, i] / processed[:, i].sum()
            score = F.softmax(processed[:, i], dim=0)

            entropy = -1 / torch.log(torch.tensor(len(domains))) * (score * torch.log(score + 1e-12)).sum()

            if entropy >= ent_thresh and processed[:, i].sum() > act_thresh:
                sum_entropy += entropy

                # --- LOG INDIVIDUAL ENTROPY ---
                class_concept_logs.append({
                    "concept_index": i,
                    "entropy": entropy.item(),
                    "scores": [s.item() for s in score],
                    "mean_acts": [val.item() for val in processed[:, i]]
                })

        invariance_val = (sum_entropy)
        if isinstance(invariance_val, torch.Tensor):
            invariance_float = invariance_val.item()
        else:
            invariance_float = invariance_val # It might already be a float (if sum_entropy was 0.0)

        # --- LOG FINAL INVARIANCE ---
        logs["final_invariance_per_class"][cls] = invariance_float
        logs["model_invariance"] += invariance_float
        # --- LOG ALL CONCEPT ENTROPIES FOR THIS CLASS ---
        logs["thresholded_concept_entropies"][cls] = class_concept_logs

        print(f"Total Thresholded Entropy (INVARIANCE) for class {cls}: {invariance_float}")


    logs["model_invariance"] /= 7
    return logs



In [ ]:
domains = ["art_painting", "cartoon", "photo"]
# domains = ["art_painting", "cartoon", "photo", "sketch"]

for ckpt in SAEs.keys():
    mean_acts = calculate_mean_activations(backbone=backbones[ckpt].to(device), sae=SAEs[ckpt], rearrange_string="n d w h -> (n w h) d", w=7, nb_concepts=2048*8, domains=domains)
    logs = calculate_invariance(mean_acts, 0.0, 0.0, domains=domains, nb_concepts=2048*8)
    save_json(logs, f"./invariances/H_ERM_ResNet_T3_step{ckpt}.json")

In [ ]:
import json
from typing import Dict, Any, Set
from collections import defaultdict, Counter

class ConceptAnalyzer:
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.data = self._load_data()

        self.iou_mode = None
        self.iou_allowed_concepts: Set[int] = set()
        # Populated by calculate_iou(); used by filter_concepts() for the IoU column
        self.concept_class_count: Dict[int, int] = {}

    # ── Data Loading ───────────────────────────────────────────────────────────

    def _load_data(self) -> Dict[str, Any]:
        try:
            with open(self.filepath, 'r') as file:
                return json.load(file)
        except FileNotFoundError:
            print(f"Error: The file {self.filepath} was not found.")
            return {}
        except json.JSONDecodeError:
            print(f"Error: The file {self.filepath} contains invalid JSON.")
            return {}

    # ── IoU Helpers ────────────────────────────────────────────────────────────

    def _get_concept_overlap_mapping(self, min_strength: float = 0.0001) -> Dict[int, set]:
        """
        Returns overlap_buckets: { n_classes -> set of concept_indices }.
        Also populates self.concept_class_count: { concept_idx -> n_classes }.
        """
        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            return {}

        concept_to_classes: Dict[int, set] = defaultdict(set)
        for class_id_str, concepts in concept_data.items():
            class_id = int(class_id_str)
            for c in concepts:
                idx      = c.get("concept_index")
                strength = sum(c.get("mean_acts", []))
                if idx is not None and strength > min_strength:
                    concept_to_classes[idx].add(class_id)

        # Cache per-concept class count for use in filter_concepts table
        self.concept_class_count = {
            idx: len(classes) for idx, classes in concept_to_classes.items()
        }

        overlap_buckets: Dict[int, set] = defaultdict(set)
        for concept_idx, classes in concept_to_classes.items():
            overlap_buckets[len(classes)].add(concept_idx)

        return overlap_buckets

    def set_iou_mode(self, target_iou: int = None, min_strength: float = 0.0001):
        """
        Restrict filter_concepts to concepts appearing in exactly `target_iou` classes.
        Pass None to disable.
        """
        self.iou_mode = target_iou
        if target_iou is not None:
            buckets = self._get_concept_overlap_mapping(min_strength)
            self.iou_allowed_concepts = buckets.get(target_iou, set())
            print(f"--- IoU Mode Enabled ---")
            print(f"  Target : {target_iou} class(es)")
            print(f"  Locked : {len(self.iou_allowed_concepts)} concept(s)\n")
        else:
            self.iou_allowed_concepts = set()
            print("--- IoU Mode Disabled ---\n")

    # ── Primary Analysis ───────────────────────────────────────────────────────

    def calculate_iou(self, min_strength: float = 0.0001) -> Dict[int, set]:
        """
        For each concept, counts how many classes it appears in and prints a
        distribution table. Populates self.concept_class_count as a side-effect
        so filter_concepts can show the IoU column without recomputing.
        """
        if not self.data:
            print("No data loaded.")
            return {}

        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            print("No 'thresholded_concept_entropies' found in the data.")
            return {}

        total_classes    = len(concept_data)
        overlap_buckets  = self._get_concept_overlap_mapping(min_strength)   # also fills concept_class_count
        total_concepts   = sum(len(v) for v in overlap_buckets.values())
        bar_width        = 30

        print(f"{'═'*62}")
        print(f"  CONCEPT CLASS OVERLAP  (min_strength > {min_strength})")
        print(f"{'═'*62}")
        print(f"  {'Classes':<10} {'# Concepts':>12}   {'Distribution'}")
        print(f"  {'─'*58}")

        for n in range(1, total_classes + 1):
            bucket     = overlap_buckets.get(n, set())
            count      = len(bucket)
            proportion = count / total_concepts if total_concepts else 0
            bar        = "█" * int(proportion * bar_width)
            label      = "class only" if n == 1 else "classes"
            print(f"  {n} {label:<12} {count:>8}   {bar} {proportion*100:.1f}%")

        print(f"  {'─'*58}")
        print(f"  {'Total':<22} {total_concepts:>8}")
        print(f"{'═'*62}\n")

        for n in range(1, total_classes + 1):
            bucket = overlap_buckets.get(n, set())
            if bucket:
                print(f"  Concepts in exactly {n} class(es) [{len(bucket)}]:")
                sorted_concepts = sorted(bucket)
                for i in range(0, len(sorted_concepts), 10):
                    print(f"    {sorted_concepts[i:i+10]}")
                print()

        return dict(overlap_buckets)

    def filter_concepts(
        self,
        entropy_min:           float = None,
        entropy_max:           float = None,
        discrimination_min:    float = None,
        discrimination_max:    float = None,
        mean_acts_sum_min:     float = None,
        mean_acts_sum_max:     float = None,
        activation_count_min:  int   = None,
        activation_count_max:  int   = None,
    ):
        """
        Filters concepts across all classes and prints a table.
        If calculate_iou() has been called beforehand, an 'IoU (#cls)' column
        showing how many classes each concept appears in is included automatically.
        """
        if not self.data:
            print("No data loaded.")
            return

        concept_data = self.data.get("thresholded_concept_entropies", {})
        if not concept_data:
            print("No 'thresholded_concept_entropies' found in the data.")
            return

        has_iou = bool(self.concept_class_count)   # True if calculate_iou() was called

        # ── Active filter summary ──────────────────────────────────────────────
        print("--- Active Filters ---")
        if self.iou_mode is not None:        print(f"  IoU Mode          == {self.iou_mode} class(es)")
        if entropy_min is not None:          print(f"  Entropy           >= {entropy_min}")
        if entropy_max is not None:          print(f"  Entropy           <= {entropy_max}")
        if discrimination_min is not None:   print(f"  Discrimination    >= {discrimination_min}")
        if discrimination_max is not None:   print(f"  Discrimination    <= {discrimination_max}")
        if mean_acts_sum_min is not None:    print(f"  Mean Acts Sum     >= {mean_acts_sum_min}")
        if mean_acts_sum_max is not None:    print(f"  Mean Acts Sum     <= {mean_acts_sum_max}")
        if activation_count_min is not None: print(f"  Activation Count  >= {activation_count_min}")
        if activation_count_max is not None: print(f"  Activation Count  <= {activation_count_max}")
        if not has_iou:
            print("  (IoU column hidden — run calculate_iou() first to enable it)")
        print()

        total_matches = 0

        for class_id, concepts in concept_data.items():
            matched = []
            for c in concepts:
                concept_idx      = c.get("concept_index")
                entropy          = c.get("entropy", 0)
                discrimination   = c.get("discrimination_score", 0)
                mean_acts_sum    = sum(c.get("mean_acts", []))
                activation_count = c.get("activation_count", 0)

                if self.iou_mode is not None and concept_idx not in self.iou_allowed_concepts:
                    continue
                if entropy_min is not None          and entropy < entropy_min:                   continue
                if entropy_max is not None          and entropy > entropy_max:                   continue
                if discrimination_min is not None   and discrimination < discrimination_min:     continue
                if discrimination_max is not None   and discrimination > discrimination_max:     continue
                if mean_acts_sum_min is not None    and mean_acts_sum < mean_acts_sum_min:       continue
                if mean_acts_sum_max is not None    and mean_acts_sum > mean_acts_sum_max:       continue
                if activation_count_min is not None and activation_count < activation_count_min: continue
                if activation_count_max is not None and activation_count > activation_count_max: continue

                matched.append(c)

            if matched:
                print(f"Class {class_id}: {len(matched)} match(es)")

                # ── Header ────────────────────────────────────────────────────
                if has_iou:
                    print(f"  {'Concept':<10} {'Entropy':<10} {'Discrimination':<18} {'Mean Acts Sum':<16} {'Act. Count':<12} {'IoU (#cls)'}")
                    print(f"  {'-'*82}")
                else:
                    print(f"  {'Concept':<10} {'Entropy':<10} {'Discrimination':<18} {'Mean Acts Sum':<16} {'Act. Count'}")
                    print(f"  {'-'*68}")

                # ── Rows ──────────────────────────────────────────────────────
                for c in matched:
                    cidx = c.get("concept_index")
                    row = (
                        f"  {cidx:<10} "
                        f"{c.get('entropy', 0):<10.4f} "
                        f"{c.get('discrimination_score', 0):<18.6f} "
                        f"{sum(c.get('mean_acts', [])):<16.2f} "
                        f"{c.get('activation_count', 0):<12}"
                    )
                    if has_iou:
                        n_classes = self.concept_class_count.get(cidx, 0)
                        row += f" {n_classes}"
                    print(row)
                print()
                total_matches += len(matched)

        print(f"Total matching concepts across all classes: {total_matches}")

In [ ]:
import os
import torch
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from einops import rearrange
import matplotlib.pyplot as plt
from torchvision import transforms
from lib.data_handlers import Load_PACS
from overcomplete.visualization.cmaps import VIRIDIS_ALPHA
from overcomplete.visualization.plot_utils import (interpolate_cv2, get_image_dimensions, show)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

dirs = {
    "art_painting"  : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\art_painting",
    "sketch"        : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\sketch",
    "photo"         : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\photo",
    "cartoon"       : r"C:\Users\sproj_ha\Desktop\DomainBed\domainbed\data\PACS\cartoon",
}

def visualize_class_on_concept(concept, class_idx, model, sae, rearrange_string, w=14, domain_roots=dirs, save_dir=None, n_images=None):
    

    model.eval()
    sae.eval()

    domain_top_images = {}  # domain -> list of (heatmap_sum, img_tensor, heatmap)
    
    # 1. Build the global sorted class list EXACTLY like the dataloader
    all_classes = set()
    for d_path in domain_roots.values():
        for entry in os.scandir(d_path):
            if entry.is_dir():
                all_classes.add(entry.name)
    sorted_classes = sorted(list(all_classes))
    
    # 2. Map the index to the actual string name
    target_class_name = sorted_classes[class_idx]

    for domain, dir_path in domain_roots.items():
        # 3. Safely build the path using the string name
        class_dir = os.path.join(dir_path, target_class_name)
        
        if not os.path.exists(class_dir):
            continue
            
        images = [Image.open(os.path.join(class_dir, path)) for path in os.listdir(class_dir)]

        if n_images is not None and n_images < len(images):
            images = random.sample(images, n_images)

        results = []  # (heatmap_sum, img_tensor, heatmap)

        for i, img in enumerate(images):
            with torch.no_grad():
                img = img.convert("RGB")
                transform = transforms.Compose([
                        transforms.Resize(256),
                        transforms.CenterCrop(224),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                            std=[0.229, 0.224, 0.225])
                ])

                img_tensor = transform(img).unsqueeze(dim=0).to(device) # Don't forget to send to device!

                x = extract_features(model, img_tensor)
                x = sae.normalizer(x)
                
                x = rearrange(x, rearrange_string)
                
                _, z = sae.encode(x)
                
                # FIX 2: Use dynamic spatial dimensions
                z = rearrange(z, '(n w h) d -> n w h d', w=w, h=w)
                
                width, height = img_tensor.shape[-1], img_tensor.shape[-2]
                
                # FIX 3: Isolate the specific 2D image, detach, move to CPU, and convert to numpy
                heatmap_2d = z[0, :, :, concept].detach().cpu().numpy()
                
                heatmap = interpolate_cv2(heatmap_2d, (width, height))
                heatmap_sum = heatmap.sum()

                if heatmap_sum > 0:
                    results.append((heatmap_sum, img_tensor.cpu(), heatmap)) # Move img_tensor back to CPU for storage/plotting

        # Sort by activation and keep top 8
        results.sort(key=lambda x: x[0], reverse=True)
        domain_top_images[domain] = results[:8]

    # Build grid: rows = domains, cols = top-8 images
    domains = list(domain_top_images.keys())
    n_domains = len(domains)
    n_cols = 8

    fig, axes = plt.subplots(n_domains, n_cols, figsize=(n_cols * 2, n_domains * 2))

    # Ensure axes is always 2D
    if n_domains == 1:
        axes = axes[np.newaxis, :]
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for row, domain in enumerate(domains):
        top = domain_top_images[domain]
        for col in range(n_cols):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(top):
                _, img_tensor, heatmap = top[col]
                # Convert image tensor to HWC numpy for display
                show(img_tensor, ax=ax)
                show(heatmap, ax=ax, cmap=VIRIDIS_ALPHA, alpha=1.0)
            if col == 0:
                ax.set_title(domain, fontsize=8, loc='left', pad=2)

    plt.suptitle(f"Class {target_class_name} — Concept {concept} | Top 8 Activations", fontsize=11, y=1.01)
    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, f"Class{class_idx}_Concept{concept}_Grid.png"), bbox_inches="tight")
        plt.close()
    else:
        plt.show()

## Discrimination Score

Verify Accuracy of Reconstructed Latents

In [ ]:
ckpt=3300
total_correct_std = 0
total_correct_sae = 0
total_samples = 0
results = {}

pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)
softmax = nn.Softmax(dim=1)
domains = ["art_painting", "cartoon", "photo", "sketch"]


for domain in domains:
    loader, _ = Load_PACS(domains=[domain])
    
    # Domain-specific metrics tracking
    d_correct_std = 0
    d_correct_sae = 0
    d_samples = 0
    
    for i, batch in enumerate(tqdm(loader, desc=f"Evaluating {domain}")):
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            
            ## Classification
            z_raw = extract_features(backbones[ckpt], x)
            preds_std = backbones[ckpt].classifier(pool(z_raw))
            preds_std = preds_std.argmax(dim=1)
            
            
            # SAE Reconstruction Pipeline
            z_norm = SAEs[ckpt].normalizer(z_raw)
            n, c, h, w = z_norm.shape 
            z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = SAEs[ckpt].encode(z_flat)
            z_recon_flat = SAEs[ckpt].decode(z_sae)
            z_recon = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon = SAEs[ckpt].normalizer.denormalize(z_recon)
            
            # Predict using reconstructed latent
            preds_sae = backbones[ckpt].classifier(pool(z_recon))
            preds_sae = preds_sae.argmax(dim=1)

            
        # Update tallies
        d_correct_std += (preds_std == y).sum().item()
        d_correct_sae += (preds_sae == y).sum().item()
        d_samples += y.size(0)
    
    # Calculate domain accuracies
    acc_std = d_correct_std / d_samples
    acc_sae = d_correct_sae / d_samples
    degradation = acc_std - acc_sae
    
    # Store result for this domain specifically
    results[domain] = {
        'std': acc_std,
        'sae': acc_sae,
        'degradation': degradation
    }
    
    # Add to global tally
    total_correct_std += d_correct_std
    total_correct_sae += d_correct_sae
    total_samples += d_samples

# --- Final Results Formatting ---
print("\n" + "="*60)
print(f"{'Domain':<15} | {'Std Acc (%)':<12} | {'SAE Acc (%)':<12} | {'Degradation':<12}")
print("-" * 60)

for dom, metrics in results.items():
    std_pct = metrics['std'] * 100
    sae_pct = metrics['sae'] * 100
    deg_pct = metrics['degradation'] * 100
    print(f"{dom:<15} | {std_pct:>9.2f}%   | {sae_pct:>9.2f}%   | {deg_pct:>8.2f}%")

print("-" * 60)

# Calculate and print overall totals
overall_std = (total_correct_std / total_samples) * 100
overall_sae = (total_correct_sae / total_samples) * 100
overall_degradation = overall_std - overall_sae

print(f"{'OVERALL':<15} | {overall_std:>9.2f}%   | {overall_sae:>9.2f}%   | {overall_degradation:>8.2f}%")
print("="*60 + "\n")

### S_disc


In [ ]:
def calculate_discrimination_scores(model, sae, dataloader, num_classes=7, nb_concepts=7680, device='cuda'):
    """
    ASSUMPTIONS:
    1. Score = P_unmasked(y_true) - P_masked(y_true).
       - Positive: Concept supports the ground truth.
       - Negative: Concept acted as a distractor/caused an error.
    2. Scores are only computed/aggregated for images where the concept's activation > 0.
    3. Aggregated over the entire dataset returning shape: (num_classes, nb_concepts).
    """
    model.eval()
    sae.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    # Pre-allocate trackers on GPU
    accumulated_scores = torch.zeros((num_classes, nb_concepts), device=device)
    activation_counts = torch.zeros((num_classes, nb_concepts), device=device)
    
    for x, y in tqdm(dataloader, desc="Calculating Concept Discrimination"):
        x, y = x.to(device), y.to(device)
        n = x.size(0)
        
        with torch.no_grad():
            # --- Base Unmasked Pass ---
            z_raw = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape
            
            z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)
            
            z_recon_flat = sae.decode(z_sae)
            z_recon = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon = sae.normalizer.denormalize(z_recon)
            
            logits_unmasked = model.classifier(pool(z_recon))
            probs_unmasked = F.softmax(logits_unmasked, dim=1)
            p_true_unmasked = probs_unmasked[torch.arange(n), y] 
            
            # --- Fast Masking Setup ---
            z_sae_img = rearrange(z_sae, '(n h w) c -> n (h w) c', n=n, h=h, w=w)
            concept_max_per_img, _ = z_sae_img.max(dim=1) # Shape: (n, nb_concepts)
            
            # Find concepts active at least once in the batch
            active_concepts_batch = torch.where(concept_max_per_img.max(dim=0)[0] > 0)[0]
            
            for c in active_concepts_batch:
                active_img_mask = concept_max_per_img[:, c] > 0
                
                if not active_img_mask.any():
                    continue
                    
                # OPTIMIZATION 1: In-place masking to save memory
                original_col = z_sae[:, c].clone()
                z_sae[:, c] = 0 
                
                # Forward pass with the single masked concept
                z_recon_flat_m = sae.decode(z_sae) # z_sae is currently masked
                z_recon_m = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
                z_recon_m = sae.normalizer.denormalize(z_recon_m)
                
                logits_masked = model.classifier(pool(z_recon_m))
                probs_masked = F.softmax(logits_masked, dim=1)
                p_true_masked = probs_masked[torch.arange(n), y]
                
                # Restore the original concept activations for the next loop iteration
                z_sae[:, c] = original_col
                
                # Calculate drop
                score_drop = p_true_unmasked - p_true_masked 
                
                # Filter to active images
                active_classes = y[active_img_mask]
                active_drops = score_drop[active_img_mask]
                
                # OPTIMIZATION 2: Vectorized GPU accumulation (No CPU sync)
                accumulated_scores[:, c].scatter_add_(0, active_classes, active_drops)
                activation_counts[:, c].scatter_add_(0, active_classes, torch.ones_like(active_drops))

    # Average the scores (clamp denominator to prevent div by zero)
    final_discrimination_scores = accumulated_scores / activation_counts.clamp(min=1)
    
    return final_discrimination_scores, activation_counts




import json

def combine_scores_to_json(existing_json_path, output_json_path, scores, counts):
    """
    Loads the existing invariance JSON, injects discrimination scores 
    AND activation counts for each concept, and saves the combined data.
    Concepts missing from the JSON are appended with default attribute values.
    """
    try:
        with open(existing_json_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return

    concept_data = data.get("thresholded_concept_entropies", {})

    num_classes, num_concepts = scores.shape

    for class_id in range(num_classes):
        class_id_str = str(class_id)

        # Ensure this class key exists in the JSON
        if class_id_str not in concept_data:
            concept_data[class_id_str] = []

        existing_concepts = concept_data[class_id_str]

        # Build a lookup of concept_index -> concept dict for fast access
        existing_index_map = {
            c["concept_index"]: c
            for c in existing_concepts
            if "concept_index" in c
        }

        for c_idx in range(num_concepts):
            disc_score = float(scores[class_id, c_idx].item())
            act_count  = int(counts[class_id, c_idx].item())

            if c_idx in existing_index_map:
                # Update the existing concept entry
                concept = existing_index_map[c_idx]
                concept["discrimination_score"] = disc_score if act_count > 0 else 0.0
                concept["activation_count"]     = act_count
            else:
                
                if act_count > 0 and disc_score != 0.0:
                    
                    new_concept = {
                        "concept_index":        c_idx,
                        "entropy":              -1,
                        "scores":               [0, 0, 0],
                        "mean_acts":            [0, 0, 0],
                        "discrimination_score": disc_score if act_count > 0 else 0.0,
                        "activation_count":     act_count,
                        "generalization_score": 0.0,
                    }
                    existing_concepts.append(new_concept)

    with open(output_json_path, 'w') as f:
        json.dump(data, f, indent=4)

    print(f"Successfully created combined JSON at: {output_json_path}")




import json
import matplotlib.pyplot as plt

def plot_concept_quality(combined_json_filepath, class_to_plot=None, min_occurrences=0):
    """
    Reads the combined JSON and plots Discrimination (X) vs. Invariance/Entropy (Y).
    If class_to_plot is provided, it only plots that specific class.
    Only plots concepts that fired in at least `min_occurrences` images.
    X-axis is fixed to [-1, 1] and Y-axis is fixed to [0, 1].
    """
    try:
        with open(combined_json_filepath, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading Combined JSON: {e}")
        return

    concept_data = data.get("thresholded_concept_entropies", {})

    plt.figure(figsize=(12, 8))
    colors = plt.cm.tab10.colors  

    if class_to_plot is not None:
        class_to_plot = str(class_to_plot)

    for class_id_str, concepts in concept_data.items():
        if class_to_plot is not None and class_id_str != class_to_plot:
            continue

        x_vals = [] 
        y_vals = [] 
        
        for concept in concepts:
            entropy = concept.get("entropy")
            disc_score = concept.get("discrimination_score")
            act_count = concept.get("activation_count", 0) # Safely default to 0 if missing

            # Check metrics AND that it meets the minimum occurrence threshold
            if (entropy is not None and 
                disc_score is not None and 
                disc_score != 0.0 and 
                act_count >= min_occurrences):
                
                x_vals.append(disc_score)
                y_vals.append(entropy)

        if x_vals and y_vals:
            plt.scatter(
                x_vals, 
                y_vals, 
                label=f"Class {class_id_str}", 
                color=colors[int(class_id_str) % len(colors)], 
                alpha=0.7,
                edgecolors='w',
                linewidth=0.5
            )

    title_suffix = f" (Class {class_to_plot})" if class_to_plot is not None else " (All Classes)"
    filter_suffix = f" | Min Occurrences: {min_occurrences}" if min_occurrences > 0 else ""
    plt.title(f"Discrimination vs. Invariance{title_suffix}{filter_suffix}", fontsize=16, fontweight='bold')
    
    plt.xlabel("Discrimination Score (Drop in Ground Truth Confidence)", fontsize=12)
    plt.ylabel("Invariance Score (Entropy across Domains)", fontsize=12)
    
    plt.xlim(-0.1, 0.1)
    plt.ylim(0, 1)
    
    plt.axvline(0, color='black', linewidth=1.5, linestyle='--')
    
    plt.grid(True, linestyle='--', alpha=0.5)
    
    if plt.gca().get_legend_handles_labels()[0]:
        plt.legend(title="Classes", bbox_to_anchor=(1.05, 1), loc='upper left')
        
    plt.tight_layout()
    plt.show()

Test 1: Only Discrimination on Train Domains

In [ ]:
domains = ["art_painting", "cartoon", "photo"]
loader, _ = Load_PACS(domains=domains)


for ckpt in backbones.keys():
    scores, counts = calculate_discrimination_scores(
        model=backbones[ckpt].to(device),
        sae=SAEs[ckpt],
        dataloader=loader,
        num_classes=7,
        nb_concepts=2048 * 8,
        device=device
    )

    combine_scores_to_json(
        existing_json_path=os.path.join(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances", f"uGEN_ERM_ResNet_T3_step{ckpt}.json"), 
        output_json_path=os.path.join(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances", f"uGEN_ERM_ResNet_T3_step{ckpt}.json"), 
        scores=scores, 
        counts=counts
    )

In [ ]:
# 2. Plot from the single file
plot_concept_quality(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step3300.json", class_to_plot=None, min_occurrences=0)

In [ ]:
# 2. Plot from the single file
plot_concept_quality(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step2100.json", class_to_plot=None, min_occurrences=0)

## A Generalization Score

In [ ]:
import json

def compute_generalization_scores(filepath: str) -> dict:
    """
    Loads a JSON file and computes:
    - generalization_score per concept (entropy * discrimination_score)
    - final_generalization_per_class per class (sum of generalization_scores)
    - model_generalization at the top level (average of class generalization scores)
    """
    with open(filepath, 'r') as f:
        data = json.load(f)

    class_generalization_totals = {}

    for class_id, concepts in data.get("thresholded_concept_entropies", {}).items():
        class_total = 0.0

        for concept in concepts:
            entropy = concept.get("entropy", 0)
            discrimination_score = concept.get("discrimination_score", 0)
            generalization_score = entropy * discrimination_score
            concept["generalization_score"] = generalization_score
            class_total += generalization_score

        class_generalization_totals[class_id] = class_total

    # Inject final_generalization_per_class
    data["final_generalization_per_class"] = class_generalization_totals

    # Compute model_generalization as average across classes
    if class_generalization_totals:
        model_generalization = sum(class_generalization_totals.values()) / len(class_generalization_totals)
    else:
        model_generalization = 0.0

    data["model_generalization"] = model_generalization

    return data


def load_and_save_generalization(input_path: str, output_path: str = None):
    result = compute_generalization_scores(input_path)

    if output_path:
        with open(output_path, 'w') as f:
            json.dump(result, f, indent=4)
        print(f"Saved enriched JSON to {output_path}")

    return result

Test 2: Include Sketch Domain

In [ ]:
load_and_save_generalization(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step3300.json", r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step3300.json")
load_and_save_generalization(r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step2100.json", r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\invariances\uGEN_ERM_ResNet_T3_step2100.json")

# Visualizations

In [ ]:
analyzer = ConceptAnalyzer(f"./invariances/uGEN_ERM_ResNet_T3_step3300.json")
analyzer.filter_concepts(
    entropy_max=0.5,
    entropy_min=0.0,
    discrimination_min=-0.001,
    discrimination_max=0.0,
    mean_acts_sum_max=None,
    mean_acts_sum_min=100,
)

### High Entropy, Highly Positive Discrimination

In [ ]:
visualize_class_on_concept(1008, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

In [ ]:
visualize_class_on_concept(2447, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

### High Entropy, Near Zero Discrimination (High Mean Activation)

In [ ]:
visualize_class_on_concept(86, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

In [ ]:
visualize_class_on_concept(1705, 1, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

### High Entropy, Negative Discrimination

In [ ]:
visualize_class_on_concept(12766, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

In [ ]:
visualize_class_on_concept(1637, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

In [ ]:
visualize_class_on_concept(3210, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

In [ ]:
visualize_class_on_concept(11462, 0, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 


# Mask Concept and Accuracy Check

In [ ]:

def masked_accuracy(
    model,
    sae,
    dataloader,
    json_filepath: str,
    generalization_score_min: float = None,
    generalization_score_max: float = None,
    entropy_min: float = None,
    entropy_max: float = None,
    discrimination_min: float = None,
    discrimination_max: float = None,
    num_classes: int = 7,
    nb_concepts: int = 7680,
    device: str = 'cuda',
) -> dict:

    # ------------------------------------------------------------------ #
    # 1. Load JSON and collect concept indices to mask (per class)         #
    # ------------------------------------------------------------------ #
    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    concepts_to_mask_per_class = defaultdict(set)
    total_masked = 0


    for class_id_str, concepts in concept_data.items():
        class_id = int(class_id_str)
        for c in concepts:
            entropy        = c.get("entropy", 0.0)
            discrimination = c.get("discrimination_score", 0.0)
            gen_score      = entropy * discrimination

            # Check all active filters — any that are set must pass
            if generalization_score_min is not None and gen_score < generalization_score_min:   continue
            if generalization_score_max is not None and gen_score > generalization_score_max:   continue
            if entropy_min is not None and entropy < entropy_min:                               continue
            if entropy_max is not None and entropy > entropy_max:                               continue
            if discrimination_min is not None and discrimination < discrimination_min:          continue
            if discrimination_max is not None and discrimination > discrimination_max:          continue

            concepts_to_mask_per_class[class_id].add(c["concept_index"])
            total_masked += 1

    # Pre-build boolean mask tensors per class
    class_concept_masks = {}
    for class_id, indices in concepts_to_mask_per_class.items():
        mask = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
        mask[list(indices)] = True
        class_concept_masks[class_id] = mask

    # ------------------------------------------------------------------ #
    # 2. Single-pass: compute both baseline and masked accuracy together   #
    # ------------------------------------------------------------------ #
    model.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    correct_baseline = torch.zeros(num_classes, device=device)
    correct_masked   = torch.zeros(num_classes, device=device)
    total_per_class  = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Masked vs Baseline Accuracy"):
            x, y = x.to(device), y.to(device)
            n = x.size(0)

            z_raw  = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape

            z_flat = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)

            # --- Baseline (no masking) ---
            z_recon_flat = sae.decode(z_sae)
            z_recon      = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon      = sae.normalizer.denormalize(z_recon)
            preds_baseline = model.classifier(pool(z_recon)).argmax(dim=1)

            # --- Masked pass ---
            z_sae_img = rearrange(z_sae, '(n h w) c -> n h w c', n=n, h=h, w=w)
            for i in range(n):
                gt_class = y[i].item()
                if gt_class in class_concept_masks:
                    z_sae_img[i, :, :, class_concept_masks[gt_class]] = 0.0

            z_sae_masked = rearrange(z_sae_img, 'n h w c -> (n h w) c')
            z_recon_flat_m = sae.decode(z_sae_masked)
            z_recon_m      = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon_m      = sae.normalizer.denormalize(z_recon_m)
            preds_masked   = model.classifier(pool(z_recon_m)).argmax(dim=1)

            # --- Accumulate ---
            for cls in range(num_classes):
                cls_mask = (y == cls)
                correct_baseline[cls] += (preds_baseline[cls_mask] == y[cls_mask]).sum()
                correct_masked[cls]   += (preds_masked[cls_mask]   == y[cls_mask]).sum()
                total_per_class[cls]  += cls_mask.sum()

    # ------------------------------------------------------------------ #
    # 3. Report                                                            #
    # ------------------------------------------------------------------ #
    per_class_baseline = (correct_baseline / total_per_class.clamp(min=1)).cpu()
    per_class_masked   = (correct_masked   / total_per_class.clamp(min=1)).cpu()
    overall_baseline   = (correct_baseline.sum() / total_per_class.sum()).item()
    overall_masked     = (correct_masked.sum()    / total_per_class.sum()).item()
    delta              = overall_masked - overall_baseline

    # print(f"\n{'='*60}")
    # print(f"{'Class':<10} {'Baseline':>12} {'Masked':>12} {'Delta':>10}")
    # print(f"{'-'*60}")
    # for cls in range(num_classes):
    #     b   = per_class_baseline[cls].item() * 100
    #     m   = per_class_masked[cls].item()   * 100
    #     d   = m - b
    #     print(f"{cls:<10} {b:>11.2f}% {m:>11.2f}% {d:>+9.2f}%")
    # print(f"{'-'*60}")
    # print(f"{'Overall':<10} {overall_baseline*100:>11.2f}% {overall_masked*100:>11.2f}% {delta*100:>+9.2f}%")
    # print(f"{'='*60}\n")

    return {
        "per_class_baseline_accuracy": {cls: per_class_baseline[cls].item() for cls in range(num_classes)},
        "per_class_masked_accuracy":   {cls: per_class_masked[cls].item()   for cls in range(num_classes)},
        "overall_baseline_accuracy":   overall_baseline,
        "overall_masked_accuracy":     overall_masked,
        "overall_delta":               delta,
        "masked_concepts":             {cls: list(idxs) for cls, idxs in concepts_to_mask_per_class.items()},
        "total_concepts_masked":       total_masked,
    }

In [ ]:
import json
from typing import List, Tuple, Optional

def _in_any_interval(value: float, intervals: Optional[List[Tuple[float, float]]]) -> bool:
    """Return True if value falls within ANY of the given [min, max] intervals.
    If intervals is None or empty, the check is skipped (always passes)."""
    if not intervals:
        return True
    return any(lo <= value <= hi for lo, hi in intervals)


def masked_accuracy_without_label_leakage(
    model,
    sae,
    dataloader,
    json_filepath: str,
    generalization_score_intervals: Optional[List[Tuple[float, float]]] = None,
    entropy_intervals:              Optional[List[Tuple[float, float]]] = None,
    discrimination_intervals:       Optional[List[Tuple[float, float]]] = None,
    num_classes: int = 7,
    nb_concepts: int = 7680,
    device: str = 'cuda',
) -> dict:

    # ------------------------------------------------------------------ #
    # 1. Load JSON and build a single global concept mask                  #
    # ------------------------------------------------------------------ #
    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    global_indices = set()

    for class_id_str, concepts in concept_data.items():
        for c in concepts:
            entropy        = c.get("entropy", 0.0)
            discrimination = c.get("discrimination_score", 0.0)
            gen_score      = entropy * discrimination

            if not _in_any_interval(gen_score,      generalization_score_intervals): continue
            if not _in_any_interval(entropy,         entropy_intervals):              continue
            if not _in_any_interval(discrimination,  discrimination_intervals):       continue

            global_indices.add(c["concept_index"])

    global_concept_mask = torch.zeros(nb_concepts, dtype=torch.bool, device=device)
    if global_indices:
        global_concept_mask[list(global_indices)] = True

    # ------------------------------------------------------------------ #
    # 2. Single-pass: baseline and masked accuracy together                #
    # ------------------------------------------------------------------ #
    model.eval()
    pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)

    correct_baseline = torch.zeros(num_classes, device=device)
    correct_masked   = torch.zeros(num_classes, device=device)
    total_per_class  = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Masked vs Baseline Accuracy"):
            x, y = x.to(device), y.to(device)
            n = x.size(0)

            z_raw  = extract_features(model, x)
            z_norm = sae.normalizer(z_raw)
            _, _, h, w = z_norm.shape

            z_flat   = rearrange(z_norm, "n c h w -> (n h w) c")
            _, z_sae = sae.encode(z_flat)

            # --- Baseline (no masking) ---
            z_recon_flat   = sae.decode(z_sae)
            z_recon        = rearrange(z_recon_flat, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon        = sae.normalizer.denormalize(z_recon)
            preds_baseline = model.classifier(pool(z_recon)).argmax(dim=1)

            # --- Masked pass: single global mask, no label needed ---
            z_sae_masked = z_sae.clone()
            z_sae_masked[:, global_concept_mask] = 0.0

            z_recon_flat_m = sae.decode(z_sae_masked)
            z_recon_m      = rearrange(z_recon_flat_m, '(n h w) c -> n c h w', n=n, h=h, w=w)
            z_recon_m      = sae.normalizer.denormalize(z_recon_m)
            preds_masked   = model.classifier(pool(z_recon_m)).argmax(dim=1)

            # --- Accumulate ---
            for cls in range(num_classes):
                cls_mask = (y == cls)
                correct_baseline[cls] += (preds_baseline[cls_mask] == y[cls_mask]).sum()
                correct_masked[cls]   += (preds_masked[cls_mask]   == y[cls_mask]).sum()
                total_per_class[cls]  += cls_mask.sum()

    # ------------------------------------------------------------------ #
    # 3. Compute metrics                                                   #
    # ------------------------------------------------------------------ #
    per_class_baseline = (correct_baseline / total_per_class.clamp(min=1)).cpu()
    per_class_masked   = (correct_masked   / total_per_class.clamp(min=1)).cpu()
    overall_baseline   = (correct_baseline.sum() / total_per_class.sum()).item()
    overall_masked     = (correct_masked.sum()    / total_per_class.sum()).item()

    return {
        "per_class_baseline_accuracy": {cls: per_class_baseline[cls].item() for cls in range(num_classes)},
        "per_class_masked_accuracy":   {cls: per_class_masked[cls].item()   for cls in range(num_classes)},
        "overall_baseline_accuracy":   overall_baseline,
        "overall_masked_accuracy":     overall_masked,
        "overall_delta":               overall_masked - overall_baseline,
        "total_concepts_masked":       len(global_indices),
        "masked_concept_indices":      sorted(global_indices),
    }

In [ ]:
# domains = ["art_painting", "cartoon", "photo", "sketch"]

# all_domain_results = {}

# for domain in domains:
#     val_loader, _ = Load_PACS(domains=[domain])

#     results = masked_accuracy(
#         model=backbones[3300].to(device),
#         sae=SAEs[3300],
#         dataloader=val_loader,
#         json_filepath="./invariances/uGEN_ERM_ResNet_T3_step3300.json",
#         entropy_intervals=None,
#         discrimination_intervals=[(-1.0, -0.01), (0.01, 1.0)],
#         generalization_score_intervals=None,   
#         num_classes=7,
#         nb_concepts=2048*8,
#         device='cuda',
#     )

#     all_domain_results[domain] = results



# # ── Cross-Domain Summary ──────────────────────────────────────────────────────
# print(f"\n{'═'*62}")
# print(f"  SUMMARY")
# print(f"{'═'*62}")
# print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
# print(f"  {'─'*54}")
# for domain, results in all_domain_results.items():
#     b    = results["overall_baseline_accuracy"] * 100
#     m    = results["overall_masked_accuracy"]   * 100
#     d    = m - b
#     sign = "+" if d >= 0 else ""
#     print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
# print(f"  {'─'*54}")

# # Averages across domains
# avg_b = sum(r["overall_baseline_accuracy"] for r in all_domain_results.values()) / len(all_domain_results) * 100
# avg_m = sum(r["overall_masked_accuracy"]   for r in all_domain_results.values()) / len(all_domain_results) * 100
# avg_d = avg_m - avg_b
# sign  = "+" if avg_d >= 0 else ""
# print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
# print(f"{'═'*62}\n")

In [ ]:
domains = ["art_painting", "cartoon", "photo", "sketch"]

all_domain_results = {}

for domain in domains:
    val_loader, _ = Load_PACS(domains=[domain])

    results = masked_accuracy_without_label_leakage(
        model=backbones[3300].to(device),
        sae=SAEs[3300],
        dataloader=val_loader,
        json_filepath="./invariances/uGEN_ERM_ResNet_T3_step3300.json",
        num_classes=7,
        nb_concepts=2048*8,
        device='cuda',
        entropy_intervals=None,
        discrimination_intervals=[(-1.0, -0.001), (0.001, 1.0)],
        generalization_score_intervals=None,   
    )

    all_domain_results[domain] = results



# ── Cross-Domain Summary ──────────────────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  SUMMARY")
print(f"{'═'*62}")
print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
print(f"  {'─'*54}")
for domain, results in all_domain_results.items():
    b    = results["overall_baseline_accuracy"] * 100
    m    = results["overall_masked_accuracy"]   * 100
    d    = m - b
    sign = "+" if d >= 0 else ""
    print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
print(f"  {'─'*54}")

# Averages across domains
avg_b = sum(r["overall_baseline_accuracy"] for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_m = sum(r["overall_masked_accuracy"]   for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_d = avg_m - avg_b
sign  = "+" if avg_d >= 0 else ""
print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
print(f"{'═'*62}\n")

In [ ]:
domains = ["art_painting", "cartoon", "photo", "sketch"]

all_domain_results = {}

for domain in domains:
    val_loader, _ = Load_PACS(domains=[domain])

    results = masked_accuracy_without_label_leakage(
        model=backbones[3300].to(device),
        sae=SAEs[3300],
        dataloader=val_loader,
        json_filepath="./invariances/uGEN_ERM_ResNet_T3_step3300.json",
        num_classes=7,
        nb_concepts=2048*8,
        device='cuda',
        entropy_intervals=[(0.0, 0.5)],
        discrimination_intervals=None,
        generalization_score_intervals=None,   
    )

    all_domain_results[domain] = results



# ── Cross-Domain Summary ──────────────────────────────────────────────────────
print(f"\n{'═'*62}")
print(f"  SUMMARY")
print(f"{'═'*62}")
print(f"  {'Domain':<20} {'Baseline':>10} {'Masked':>10} {'Δ':>8}")
print(f"  {'─'*54}")
for domain, results in all_domain_results.items():
    b    = results["overall_baseline_accuracy"] * 100
    m    = results["overall_masked_accuracy"]   * 100
    d    = m - b
    sign = "+" if d >= 0 else ""
    print(f"  {domain.replace('_', ' ').title():<20} {b:>9.2f}% {m:>9.2f}% {sign}{d:>6.2f}%")
print(f"  {'─'*54}")

# Averages across domains
avg_b = sum(r["overall_baseline_accuracy"] for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_m = sum(r["overall_masked_accuracy"]   for r in all_domain_results.values()) / len(all_domain_results) * 100
avg_d = avg_m - avg_b
sign  = "+" if avg_d >= 0 else ""
print(f"  {'Average':<20} {avg_b:>9.2f}% {avg_m:>9.2f}% {sign}{avg_d:>6.2f}%")
print(f"{'═'*62}\n")

In [ ]:
analyzer = ConceptAnalyzer(f"./invariances/uGEN_ERM_ResNet_T3_eval_step3300.json")
classes = analyzer.calculate_iou() # --> EXPLORE THIS DIRECTION FURTHER.

print("done")

In [ ]:
import json
import numpy as np
from scipy import stats
from typing import Optional

def compute_activation_discrimination_correlation(
    json_filepath: str,
    class_ids: Optional[list] = None,
    mean_acts_aggregation: str = 'sum',  # 'sum', 'max', or 'mean' across the per-class mean_acts list
) -> dict:
    """
    Computes the correlation between mean activation strength and discrimination
    score for all concepts in each class.

    Args:
        json_filepath:          Path to the concept entropy JSON file.
        class_ids:              List of class IDs (as ints) to compute correlation for.
                                If None, computes for all classes in the file.
        mean_acts_aggregation:  How to collapse the per-subclass mean_acts list into
                                a single scalar. One of 'sum', 'max', 'mean'.

    Returns:
        A dict keyed by class_id (int) with:
            - pearson_r, pearson_p
            - spearman_r, spearman_p
            - n_concepts
            - mean_acts_values       (list of scalars used)
            - discrimination_values  (list of scalars used)
    """

    agg_fn = {
        'sum':  np.sum,
        'max':  np.max,
        'mean': np.mean,
    }.get(mean_acts_aggregation)
    if agg_fn is None:
        raise ValueError(f"mean_acts_aggregation must be 'sum', 'max', or 'mean', got '{mean_acts_aggregation}'")

    with open(json_filepath, 'r') as f:
        data = json.load(f)

    concept_data = data.get("thresholded_concept_entropies", {})

    results = {}

    for class_id_str, concepts in concept_data.items():
        class_id = int(class_id_str)

        if class_ids is not None and class_id not in class_ids:
            continue

        activation_strengths = []
        discrimination_scores = []

        for c in concepts:
            mean_acts      = c.get("mean_acts", [])
            discrimination = c.get("discrimination_score", None)

            if not mean_acts or discrimination is None:
                continue

            activation_strengths.append(float(agg_fn(mean_acts)))
            discrimination_scores.append(float(discrimination))

        n = len(activation_strengths)

        if n < 3:
            # Not enough points for a meaningful correlation
            results[class_id] = {
                "pearson_r":             None,
                "pearson_p":             None,
                "spearman_r":            None,
                "spearman_p":            None,
                "n_concepts":            n,
                "mean_acts_values":      activation_strengths,
                "discrimination_values": discrimination_scores,
                "note":                  "Too few concepts for correlation.",
            }
            continue

        pearson_r,  pearson_p  = stats.pearsonr(activation_strengths,  discrimination_scores)
        spearman_r, spearman_p = stats.spearmanr(activation_strengths, discrimination_scores)

        results[class_id] = {
            "pearson_r":             pearson_r,
            "pearson_p":             pearson_p,
            "spearman_r":            spearman_r,
            "spearman_p":            spearman_p,
            "n_concepts":            n,
            # "mean_acts_values":      activation_strengths,
            # "discrimination_values": discrimination_scores,
        }

    return results

In [ ]:
compute_activation_discrimination_correlation("./invariances/uGEN_ERM_ResNet_T3_step3300.json", mean_acts_aggregation='sum')

# Further Analysis

Goals
1. Identify "what" exists in each of the Quadrants
2. Verify the trend across MMD, and a ViT model of choice
3. Verify the SGen Score and whether it persists for other checkpoints and USAEs (ALL CKPTS and MMD)
4. Move everything to lib so I can do further analysis in a cleaner way
5. Verify concept IoU, Why on earth does a flat out P hide not work

-- What exists in the Quadrants

In [ ]:
analyzer = ConceptAnalyzer(f"./invariances/uGEN_ERM_ResNet_T3_step3300.json")
#iou = analyzer.calculate_iou()

In [ ]:
analyzer.set_iou_mode(target_iou=None)
analyzer.filter_concepts(
    entropy_max=0.5,
    entropy_min=None,
    discrimination_min=0.001,
    discrimination_max=None,
    mean_acts_sum_max=None,
    mean_acts_sum_min=100,
)

In [ ]:
for i in range(7):
    visualize_class_on_concept(15483, i, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=r"C:\Users\sproj_ha\Desktop\SGen_Vision_Interp\Vision_Interp\quadrant_figures\ERM_ResNet_T3\15483 - 1ClassHighInvariance", n_images=None, domain_roots=dirs) 

In [ ]:
for i in range(7):
    visualize_class_on_concept(15483, i, backbones[3300].to(device), sae=SAEs[3300], rearrange_string='n c w h -> (n w h) c', w=7, save_dir=None, n_images=None, domain_roots=dirs) 

Tradeoff Curve

In [ ]:
import json
import math
from typing import Dict, Any


def compute_discrimination_entropy(json_path: str) -> None:
	# Load JSON
	with open(json_path, "r") as f:
		data: Dict[str, Any] = json.load(f)

	concept_entries_by_class = data.get("thresholded_concept_entropies", {})
	if not isinstance(concept_entries_by_class, dict):
		raise ValueError("JSON field 'thresholded_concept_entropies' must be a dict")

	# Determine class keys and ensure a consistent ordering
	try:
		class_keys = sorted(concept_entries_by_class.keys(), key=lambda k: int(k))
	except (TypeError, ValueError):
		# Fallback: lexical sort if keys are not numeric strings
		class_keys = sorted(concept_entries_by_class.keys())

	# Collect all unique concept indices across all classes
	all_concept_indices = set()
	for class_key in class_keys:
		for entry in concept_entries_by_class.get(class_key, []):
			idx = entry.get("concept_index")
			if idx is not None:
				all_concept_indices.add(idx)

	# Compute discrimination entropy per concept
	concept_entropies: Dict[Any, float] = {}
	for concept_idx in all_concept_indices:
		activations_per_class = []
		for class_key in class_keys:
			entries = concept_entries_by_class.get(class_key, [])
			class_activation = 0.0
			for entry in entries:
				if entry.get("concept_index") == concept_idx:
					mean_acts = entry.get("mean_acts", 0.0)
					if isinstance(mean_acts, list):
						class_activation = float(sum(mean_acts))
					else:
						class_activation = float(mean_acts)
					break
			activations_per_class.append(class_activation)

		total_activation = sum(activations_per_class)
		if total_activation <= 0.0:
			entropy = 0.0
		else:
			probs = [a / total_activation for a in activations_per_class]
			entropy = -sum(p * math.log(p) for p in probs if p > 0.0)

		concept_entropies[concept_idx] = float(entropy)

	# Attach discrimination_entropy to every occurrence of each concept
	for class_key in class_keys:
		entries = concept_entries_by_class.get(class_key, [])
		for entry in entries:
			idx = entry.get("concept_index")
			if idx in concept_entropies:
				entry["discrimination_entropy"] = concept_entropies[idx]

	# Save updated JSON back to the same file
	with open(json_path, "w") as f:
		json.dump(data, f, indent=4)


if __name__ == "__main__":
	# Example usage; adjust the path(s) as needed
	example_path = "invariances/uGEN_ERM_ResNet_T3_step3300.json"
	compute_discrimination_entropy(example_path)


In [ ]:
compute_discrimination_entropy("invariances/uGEN_ERM_ResNet_T3_step3300.json")

In [ ]:
def plot_entropy_vs_discrimination_entropy(
    json_path: str,
    class_key: str,
    normalize_disc_entropy: bool = True,
    min_activation_count: int = 0,      # ← NEW
    max_activation_count: Optional[int] = None,  # ← NEW
    figsize: tuple = (9, 7),
    save_path: Optional[str] = None,
) -> None:
    """
    For a single class, scatter-plots concept entropy (x-axis) against
    discrimination entropy (y-axis).

    Parameters
    ----------
    json_path             : Path to the JSON file produced by compute_discrimination_entropy.
    class_key             : The class to visualise (e.g. "0", "1", …).
    normalize_disc_entropy: If True, normalises discrimination entropy to [0, 1]
                            by dividing by ln(K).
    min_activation_count  : Only include concepts with activation_count >= this value.
    max_activation_count  : Only include concepts with activation_count <= this value.
                            If None, no upper bound is applied.
    figsize               : Figure size passed to matplotlib.
    save_path             : If given, saves the figure to this path instead of showing it.
    """

    # ── Load ──────────────────────────────────────────────────────────────────
    with open(json_path, "r") as f:
        data: Dict[str, Any] = json.load(f)

    concept_entries_by_class = data.get("thresholded_concept_entropies", {})
    entries = concept_entries_by_class.get(class_key)
    if entries is None:
        raise KeyError(f"Class key '{class_key}' not found in JSON.")

    n_classes = len(concept_entries_by_class)

    # ── Extract & filter per-concept values ───────────────────────────────────
    concept_indices   = []
    entropies         = []
    disc_entropies    = []
    activation_counts = []
    n_filtered        = 0

    for entry in entries:
        ent   = entry.get("entropy")
        d_ent = entry.get("discrimination_entropy")
        count = entry.get("activation_count", 0)

        if ent is None or d_ent is None:
            continue

        # ── Activation count filter ──────────────────────────────────────────
        if count < min_activation_count:
            n_filtered += 1
            continue
        if max_activation_count is not None and count > max_activation_count:
            n_filtered += 1
            continue

        if normalize_disc_entropy and n_classes > 1:
            d_ent = d_ent / math.log(n_classes)

        concept_indices.append(entry.get("concept_index", "?"))
        entropies.append(ent)
        disc_entropies.append(d_ent)
        activation_counts.append(count)

    if not entropies:
        raise ValueError(
            f"No concepts remain after filtering for class '{class_key}'. "
            f"({n_filtered} concept(s) were filtered out.) "
            f"Try relaxing min/max_activation_count."
        )

    # ── Colour by activation count ────────────────────────────────────────────
    counts  = np.array(activation_counts, dtype=float)
    norm    = plt.Normalize(vmin=counts.min(), vmax=counts.max())
    colours = cm.viridis(norm(counts))

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(
        entropies,
        disc_entropies,
        c=colours,
        s=60,
        alpha=0.85,
        edgecolors="white",
        linewidths=0.4,
        zorder=3,
    )

    for x, y, idx in zip(entropies, disc_entropies, concept_indices):
        ax.annotate(
            str(idx),
            xy=(x, y),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
            color="#333333",
            zorder=4,
        )

    # ── Reference line ────────────────────────────────────────────────────────
    ax.axhline(
        y=1.0 if normalize_disc_entropy else math.log(n_classes),
        color="tomato", linestyle="--", linewidth=1.0,
        label=f"Max disc. entropy ({'1.0 (normalised)' if normalize_disc_entropy else f'ln({n_classes})'})",
    )

    # ── Subtitle showing filter state ─────────────────────────────────────────
    filter_parts = []
    if min_activation_count > 0:
        filter_parts.append(f"min count ≥ {min_activation_count}")
    if max_activation_count is not None:
        filter_parts.append(f"max count ≤ {max_activation_count}")
    filter_str = f"  [{', '.join(filter_parts)}  |  {n_filtered} concept(s) hidden]" if filter_parts else ""

    disc_label = (
        "Discrimination Entropy (normalised, 0 = class-specific, 1 = uniform)"
        if normalize_disc_entropy
        else "Discrimination Entropy (nats)"
    )

    ax.set_xlabel("Concept Entropy", fontsize=11)
    ax.set_ylabel(disc_label, fontsize=11)
    ax.set_title(
        f"Concept Entropy vs Discrimination Entropy — Class {class_key}\n"
        f"{filter_str}",
        fontsize=12,
    )
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, (1.05 if normalize_disc_entropy else math.log(n_classes) * 1.05))
    ax.legend(fontsize=9)

    sm = cm.ScalarMappable(cmap="viridis", norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Activation Count", fontsize=10)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Figure saved to {save_path}")
    else:
        plt.show()


plot_entropy_vs_discrimination_entropy(
    json_path="invariances/uGEN_ERM_ResNet_T3_step3300.json",
    class_key="0",
    normalize_disc_entropy=True,
    min_activation_count=100,   # ignore rarely-firing concepts
)